> **AI-generated documentation notice.** Every docstring and inline comment in this notebook was written by Claude Opus 4.7 (via Claude Code) from our code, then read and verified by us. The computation, methods, and results are entirely the team's own work. See `doc/FINAL/SIMC2026_AI_Usage_Log.pdf`, Entry 1.

# SIMC 2026 — Endeavour Challenge: combined solution notebook

This notebook stitches together the code for all parts of the challenge that have a
computational component, in solution order, so the whole pipeline runs top-to-bottom from
one kernel:

- **Task 1 — Markov-matrix model** (numerical validation, parts f & h). Self-contained;
  reads no external data.
- **Task 2 — Power Method & SVD** is **analytical only** (proofs of convergence, the
  spectral-gap rate, the X^T X / X X^T equivalence, and a worked 3×2 SVD by hand). It has
  **no code**; the full worked solution is in the preliminary report. A short marker cell
  sits where Task 2 would go.
- **Task 3 — Design / covariance / Gram matrices** (PCA & clustering, parts a–g). Reads
  `data/input/mystery_matrix1.npy`; writes figures to `data/output/TASK 3/`.

**How to run.** Execute cells in order. Task 3 auto-locates the repo root, so the notebook
works whether launched from `src/` or the repo root. The canonical, individually-maintained
notebooks remain `src/task1/task1.ipynb` and `src/task_3.ipynb`; this file is the merged view.


In [ ]:
# ===========================================================
# Part 1f — numerical validation of parts (d) and (e)
# ===========================================================
'''Purpose. Confirm the closed-form solution P(t) = c1*v1 + c2*(1-eps)^t * v2 (part 1d)
and its limit P(t) -> (0,1) (part 1e) by checking them against a brute-force iteration of
the recursion that uses no analytical input.

Method (two independent tracks):
  - Track A (ground truth): iterate P(t+1) = M @ P(t) for T steps.
  - Track B (closed form): decompose P(0) in the eigenbasis {v1, v2}, then evaluate the
    scalar-power formula for all t at once via broadcasting.
  The eigenpairs are taken from part 1b (lambda1=1 persistent, lambda2=1-eps decaying); the
  coefficients c1, c2 are obtained by solving the 2x2 linear system P(0) = c1*v1 + c2*v2,
  not hard-coded, so the cell also re-derives the c1=1, c2=1/2 of part 1c.

Validation & outputs. max|A - B| must be < 1e-10 (asserted, so a broken formula fails
loudly). P(T) is printed to show convergence to (0,1). Defines t, c1, c2, v1, v2, P_num,
which the 1h cells reuse — run this cell first.'''
import numpy as np

# --- Parameters ---
eps = 0.1                                 # conversion rate per generation
p0  = np.array([0.5, 0.5])                # initial state (p2 = 1 - p1)
T   = 100                                 # generations to simulate

# Markov matrix; lower-triangular, column-stochastic.
M = np.array([[1 - eps, 0.0],
              [eps,     1.0]])

# --- Stage 1: direct simulation ---
# Ground truth: iterate the recursion with no analytical input.
P_num = np.zeros((T + 1, 2))
P_num[0] = p0
for t_step in range(T):
    P_num[t_step + 1] = M @ P_num[t_step]

# --- Stage 2: closed form via eigenmode decomposition ---
# Eigenvectors from 1b (lambda_1 = 1, lambda_2 = 1 - eps).
v1 = np.array([0,  1])                    # persistent mode
v2 = np.array([1, -1])                    # decaying   mode

# Solve P(0) = c1*v1 + c2*v2 for the coefficients in the eigenbasis.
c1, c2 = np.linalg.solve(np.column_stack([v1, v2]), p0)
print(f"c1 = {c1},  c2 = {c2}")           # expect c1 = 1, c2 = 0.5

# Evaluate the closed form for all t at once via broadcasting.
t = np.arange(T + 1)
P_closed = c1 * v1 + c2 * ((1 - eps) ** t)[:, None] * v2

# --- Stage 3: validation ---
# Both arrays must agree to machine precision; the assertion catches any
# future edit that breaks the closed form.
max_err = np.max(np.abs(P_num - P_closed))
print(f"max |numeric - closed form| = {max_err:.2e}")
assert max_err < 1e-10, "closed form disagrees with simulation -- check 1d"

# Asymptotic check (validates 1e): P(T) sits near [0, 1] = nu_1.
print(f"P(T={T}) = {P_num[-1]}   (should approach [0, 1])")

In [ ]:
# ===== Shared plotting setup (used by both 1f and 1h) =====
# Notebook-friendly Matplotlib setup: prefer inline; fallback to Agg if unavailable
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    import matplotlib
    try:
        matplotlib.use('Agg')
    except Exception:
        pass
import matplotlib
print('matplotlib', matplotlib.__version__, 'backend', matplotlib.get_backend())

In [ ]:
# ===== Part 1f — plot (linear + log) =====
import matplotlib.pyplot as plt
from pathlib import Path

# Match the report's serif typesetting.
plt.rcParams.update({"font.family": "serif", "font.size": 11})

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# --- Panel A: linear-scale trajectories ---
# Dotted horizontal lines mark the asymptotic values, making the limit
# in 1e visible at a glance.
ax = axes[0]
ax.plot(t, P_num[:, 0], "o-", color="tab:blue",   ms=3, lw=1,
        label=r"$p_1(t)$ (type 1)")
ax.plot(t, P_num[:, 1], "o-", color="tab:orange", ms=3, lw=1,
        label=r"$p_2(t)$ (type 2)")
ax.axhline(0, color="tab:blue",   ls=":", lw=1, alpha=0.6)
ax.axhline(1, color="tab:orange", ls=":", lw=1, alpha=0.6)
ax.set_xlabel(r"generation $t$")
ax.set_ylabel("fractional population")
ax.set_title(rf"Population trajectories ($\epsilon = {eps}$)")
ax.set_ylim(-0.02, 1.05)
ax.legend(loc="center right")
ax.grid(alpha=0.3)

# --- Panel B: log-scale; numeric vs closed form ---
# A straight line on semilog-y proves the decay is geometric with rate
# (1 - eps); numeric dots sitting on the line validate the closed form.
theory_p1 = 0.5 * (1 - eps) ** t          # (1/2)(1-eps)^t

ax = axes[1]
ax.semilogy(t, theory_p1,   "-",  color="black",    lw=1.2,
            label=r"theory: $\frac{1}{2}(1-\epsilon)^t$")
ax.semilogy(t, P_num[:, 0], "o",  color="tab:blue", ms=4,
            label=r"numeric $p_1(t)$")
ax.set_xlabel(r"generation $t$")
ax.set_ylabel(r"$p_1(t)$  (log scale)")
ax.set_title("Geometric decay of type-1: numeric vs closed form")
ax.legend()
ax.grid(which="both", alpha=0.3)

fig.tight_layout()

# Save under data/output/ (repo convention). Walk up to the repo root so
# the path is correct regardless of the kernel's working directory.
root = Path.cwd().resolve()
while not (root / "data").is_dir() and root != root.parent:
    root = root.parent
out_dir = root / "data" / "output"
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "task1f_trajectories.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ===========================================================
# Part 1h — numerical validation of part (g): time-varying eps(t)
# ===========================================================
'''Purpose. Validate the part-(g) generalisation to a decaying rate eps(t)=eps0*exp(-t/t0),
and its qualitative consequence: type-1 is NOT fully depleted but freezes at a positive
residue. This is the key correction over the notes, which had (1-eps(t))^t instead of the
cumulative product.

Why a product, not a power. With eps time-varying the update matrix M(t) changes every step,
but its eigenvectors v1, v2 are eps-independent (same as 1b) and only lambda2(t)=1-eps(t)
varies. Acting on P(0)=c1*v1+c2*v2, the v2 mode collects ONE factor (1-eps(s)) per step, so
the geometric power (1-eps)^t of 1d becomes the cumulative product Pi(t)=prod_{s<t}(1-eps(s)).

Method (mirrors 1f). Track A: iterate with a freshly built M(s) each step. Track B: the
closed form c1*v1 + c2*Pi(t)*v2, with Pi computed by np.cumprod (Pi(0)=1, the empty product).
Reuses t, c1, c2, v1, v2 from the 1f cell.

Validation & outputs. max|A - B| < 1e-10 (asserted). Because sum_s eps(s) < infinity, Pi(inf)
is strictly positive, so p1 plateaus at c2*Pi(inf); we print this residue alongside the
small-eps / large-t0 estimate (1/2)e^(-eps0*t0) for a sanity check. Defines P_num_g, Pi,
plateau_p1 for the 1h plot cell.'''

eps0  = 0.5                               # initial conversion rate
t0    = 10.0                              # decay timescale of eps(t)
eps_t = eps0 * np.exp(-t / t0)            # eps(s) for s = 0..T  (t from 1f cell)

# --- Stage 1: direct simulation with a time-varying matrix ---
P_num_g = np.zeros((T + 1, 2))
P_num_g[0] = p0
for s in range(T):
    e = eps_t[s]
    M_s = np.array([[1 - e, 0.0],
                    [e,     1.0]])
    P_num_g[s + 1] = M_s @ P_num_g[s]

# --- Stage 2: closed form via the cumulative product Pi(t) ---
# Pi(0) = 1 (empty product); Pi(t) = prod_{s=0}^{t-1} (1 - eps(s)).
Pi = np.concatenate(([1.0], np.cumprod(1 - eps_t[:-1])))
P_closed_g = c1 * v1 + c2 * Pi[:, None] * v2   # reuse c1=1, c2=0.5, v1, v2 from 1f

# --- Stage 3: validation ---
max_err_g = np.max(np.abs(P_num_g - P_closed_g))
print(f"max |numeric - closed form| (1h) = {max_err_g:.2e}")
assert max_err_g < 1e-10, "1g closed form disagrees with simulation -- check the product"

# --- Asymptotic check (validates 1g): a frozen, non-zero residue ---
plateau_p1     = 0.5 * Pi[-1]             # actual residue p1(inf) = c2 * Pi(inf)
plateau_approx = 0.5 * np.exp(-eps0 * t0) # guide's small-eps / large-t0 estimate
print(f"P(T={T}) = {P_num_g[-1]}   (p1 freezes above 0, unlike 1f)")
print(f"residue p1(inf) = {plateau_p1:.6f}   "
      f"(approx (1/2)e^(-eps0*t0) = {plateau_approx:.6f})")

In [ ]:
# ===== Part 1h — plot: time-varying eps(t) vs constant eps (1f) =====
import matplotlib.pyplot as plt
from pathlib import Path

fig2, axes2 = plt.subplots(1, 2, figsize=(11, 4))

# --- Panel A: linear scale, both scenarios overlaid ---
# Faded = constant eps (1f, for reference); solid = time-varying eps(t) (1h).
ax = axes2[0]
ax.plot(t, P_num[:, 0],   "-",  color="tab:blue",   lw=1, alpha=0.35,
        label=r"$p_1$, const $\epsilon$ (1f)")
ax.plot(t, P_num[:, 1],   "-",  color="tab:orange", lw=1, alpha=0.35,
        label=r"$p_2$, const $\epsilon$ (1f)")
ax.plot(t, P_num_g[:, 0], "o-", color="tab:blue",   ms=3, lw=1.2,
        label=r"$p_1(t)$, $\epsilon(t)$ (1h)")
ax.plot(t, P_num_g[:, 1], "o-", color="tab:orange", ms=3, lw=1.2,
        label=r"$p_2(t)$, $\epsilon(t)$ (1h)")
ax.axhline(plateau_p1,     color="tab:blue",   ls=":", lw=1, alpha=0.7)
ax.axhline(1 - plateau_p1, color="tab:orange", ls=":", lw=1, alpha=0.7)
ax.set_xlabel(r"generation $t$")
ax.set_ylabel("fractional population")
ax.set_title(rf"Time-varying vs constant $\epsilon$  ($\epsilon_0={eps0},\ t_0={t0:g}$)")
ax.set_ylim(-0.02, 1.05)
ax.legend(loc="center right", fontsize=8)
ax.grid(alpha=0.3)

# --- Panel B: log scale -- the headline contrast ---
# Constant eps: p1 -> 0 (straight line on semilog-y). Time-varying eps(t):
# p1 *plateaus* at the frozen residue once eps(t) effectively switches off.
ax = axes2[1]
ax.semilogy(t, P_num[:, 0],   "-",  color="gray",     lw=1.2,
            label=r"const $\epsilon$: $p_1 \to 0$")
ax.semilogy(t, P_num_g[:, 0], "o",  color="tab:blue", ms=4,
            label=r"$\epsilon(t)$: $p_1(t)$")
ax.axhline(plateau_p1, color="tab:red", ls="--", lw=1,
           label=r"frozen residue $p_1(\infty)=\frac{1}{2}\Pi(\infty)$")
ax.set_xlabel(r"generation $t$")
ax.set_ylabel(r"$p_1(t)$  (log scale)")
ax.set_title(r"Decaying $\epsilon(t)$ freezes $p_1$ at a positive residue")
ax.legend(fontsize=8)
ax.grid(which="both", alpha=0.3)

fig2.tight_layout()

# Save under data/output/ (same repo-root resolution as the 1f cell).
root = Path.cwd().resolve()
while not (root / "data").is_dir() and root != root.parent:
    root = root.parent
out_dir = root / "data" / "output"
out_dir.mkdir(parents=True, exist_ok=True)
fig2.savefig(out_dir / "task1h_trajectories.png", dpi=150, bbox_inches="tight")
plt.show()


# Task 2 — The Power Method and Singular Value Decomposition  *(analytical — no code)*

Task 2 is solved entirely on paper; there is no notebook code to run. For completeness the
results proved in the report are:

1. **(a)** The Power Method $x_{k+1}=Xx_k/\lVert Xx_k\rVert$ converges to $\pm v_1$ whenever
   $x_0$ has a nonzero component along $v_1$ and $|\lambda_1|>|\lambda_2|$.
2. **(b)** Convergence is geometric at rate $|\lambda_2/\lambda_1|$ (the spectral gap).
3. **(c)** A vanishing gap ($|\lambda_1|=|\lambda_2|$) is a structural obstruction — the
   iterate oscillates or drifts in a degenerate eigenspace; near-degeneracy compounds with
   $10^{-16}$ round-off.
4. **(d)** $X^\top X$ and $XX^\top$ are symmetric PSD and share their nonzero eigenvalues
   $\sigma_i^2$ — the fact Task 3(f)/(g) rely on to connect the covariance and Gram views.
5. **(e)** Power-Method SVD algorithm (work with the smaller of $X^\top X$, $XX^\top$) plus a
   worked $3\times2$ example.

See **§2 of the preliminary report** for the full proofs.


# TASK 3 - THE DESIGN MATRIX, COVARIANCE MATRIX AND THE GRAM MATRIX

In [ ]:
#Module purpose. Question 3a, part (i): detect and remove the corrupted "empty vial" measurements from the spectroscopy design matrix mystery_matrix1.npy.

'''Data contract / assumptions.
  - The file loads as a float64 array of shape (3401, 1000). The problem defines the design matrix X with samples as rows (N = 1000 vials) and features as
  columns (frequencies). The on-disk array is therefore transposed relative to that convention: its 1000 axis is the samples. We transpose once, up front,
  so that downstream code follows the report's row = sample convention. (Note: the problem text states M = 3041 features; the actual file has 3401 columns.
  We use the array's true dimension and flag the discrepancy rather than hard-coding either number.)
  - All entries are non-negative intensities in [0, ~0.85]; there are no NaN/inf values. This is asserted, not assumed silently.

  Detection method. Each measurement is reduced to a single scalar, its row norm ‖xᵢ‖₂ = total signal energy across all frequencies. A real sample has
  absorption peaks (high norm); an empty vial is low background noise everywhere (low norm). The norm distribution is bimodal with an empty gap, so a hard
  threshold placed anywhere inside the gap separates the two populations exactly.

  Threshold choice & robustness. The cutoff is 3.0, but the result is invariant to any threshold in [2.0, 4.0] because the gap is genuinely empty. This
  invariance is checked explicitly — it is the evidence that the cut is principled rather than arbitrary.

  Validation strategy. Two independent checks: (1) summary statistics (mean/max/per-channel spread) must show the removed group is flat and low while the
  kept group has structure; (2) a visual overlay of removed vs kept spectra must show flat noise vs clear peaks.

  Outputs. T (number removed, expected 40), a boolean mask empty, the cleaned matrix S_clean (960 × 3401), and a figure q3a_part_i.png. The cleaned matrix
  is the input to 3b onward.'''



In [ ]:
#Cell 1 — Imports

import os
import numpy as np
import matplotlib.pyplot as plt

# Run from the repository root so the relative data/ and data/output/TASK 3/ paths
# below resolve, whether this notebook is launched from src/ or the repo root.
for _ in range(5):
    if os.path.exists("data/input/mystery_matrix1.npy"):
        break
    os.chdir("..")
DATA_PATH = "data/input/mystery_matrix1.npy"


In [ ]:
#Cell 2 — Load and sanity-check the raw array

# Load the raw design-matrix file.
# NOTE: the file is stored as (3401, 1000). The problem convention is
# samples-as-rows with N = 1000 vials, so the file is effectively X^T.
raw = np.load(DATA_PATH)

print("raw shape :", raw.shape)
print("dtype     :", raw.dtype)
print("min / max :", raw.min(), raw.max())
print("NaN / inf :", np.isnan(raw).any(), np.isinf(raw).any())

# The problem states M = 3041 features; the file has 3401. We trust the file
# and key off the axis whose length is 1000 (the samples).
assert 1000 in raw.shape, "expected one axis to be the 1000 samples"



In [ ]:
#Cell 3 — Orient to "samples as rows"

# Transpose if necessary so that rows = samples (vials), cols = frequencies.
S = raw.T if raw.shape[0] != 1000 else raw
N, M = S.shape
print(f"design matrix oriented: {N} samples (rows) x {M} frequencies (cols)")



In [ ]:
#Cell 4 — Reduce each measurement to its row norm

# Row norm = total signal energy of one vial across all frequencies.
# Empty vials (background noise only) have small norm; real samples have peaks.
norms = np.linalg.norm(S, axis=1)
print("row-norm summary")
print("  min   :", norms.min())
print("  max   :", norms.max())
print("  mean  :", norms.mean())
print("  median:", np.median(norms))



In [ ]:
#Cell 5 — Inspect the distribution (find the gap)
# A text histogram makes the bimodal structure and the empty gap obvious.
hist, edges = np.histogram(norms, bins=40)
for h, e in zip(hist, edges[:-1]):
  print(f"  {e:5.2f} : {'#' * h} {h}")
# A tight clump of 40 sits at ~1.58-1.76, then an empty gap until ~4.15.



In [ ]:
#Cell 6 — Threshold and confirm robustness
#The empty-vial clump is well below the real samples. Any threshold inside the
#gap gives the same count, which is the evidence the cut is not arbitrary.
for thr in (2.0, 3.0, 4.0):
    print(f"  rows with norm < {thr}: {(norms < thr).sum()}")  # all 40

THRESHOLD = 3.0
empty = norms < THRESHOLD          # boolean mask of corrupted measurements
real  = ~empty

T = int(empty.sum())
print(f"\nT = {T} corrupted (empty-vial) measurements")
print("their norm range:", norms[empty].min(), "->", norms[empty].max())
print("remaining measurements:", int(real.sum()))   # 960



In [ ]:
#Cell 7 — Validate the removal (statistics)
# Removed group should be flat & low; kept group should have structure (peaks).
def describe(mask, label):
  """Print summary statistics (per-channel mean, max, mean per-row std) for the rows of S picked out by the boolean `mask`, tagged with `label`; used to contrast the removed (flat, low) vials with the kept (peaked) ones."""
  print(f"{label:12s}: mean/chan = {S[mask].mean():.4f}  "
      f"max = {S[mask].max():.4f}  "
      f"per-channel std = {S[mask].std(axis=1).mean():.4f}")
describe(empty, "empty (rm)")
describe(real,  "real (keep)")



In [ ]:
#Cell 8 — Validate the removal (figure)

# Single panel: the row-norm distribution with the empty-vial threshold marked.
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(norms, bins=60, color="steelblue", edgecolor="k", linewidth=0.3)
ax.axvline(THRESHOLD, color="red", ls="--", label=f"threshold = {THRESHOLD}")
ax.set_xlabel(r"row norm  $\|x_i\|$")
ax.set_ylabel("count")
ax.set_title(f"Row-norm distribution (1000 measurements)\n"
             f"{T} empty vials isolated below the gap")
ax.legend()
plt.tight_layout()
import os
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3a_part_i.png", dpi=120)
plt.show()

In [ ]:
#Cell 9 — Produce the cleaned matrix for downstream parts

  # Cleaned design matrix after removing the T empty-vial measurements.
  # This (960 x 3401) array is the input to 3a(ii) and 3b onward.
S_clean = S[real]
print("cleaned design matrix:", S_clean.shape)   # (960, 3401)

In [ ]:
#Module purpose. Question 3a, part (ii): find the "other error" beyond the empty vials.

'''The other error -- high-norm outliers.
  Inputs. the 1000 row norms and the empty-vial mask from part (i).
  Method. Sort the norms and inspect the UPPER tail. The largest gap above the median isolates a
    second, detached group of 36 vials (norm ~8.9), separated from the bulk by a gap about five
    times wider than any internal gap and forming the contiguous block of rows 924-959.
  Honest caveat. Such a coherent group could be a genuinely strong concoction rather than an
    error, so we (a) gather independent evidence in the next cell (it forms one isolated cluster
    whose removal drops the cluster count by exactly one), and (b) keep the removal reversible.
  Outputs. boolean mask `high`; the fully cleaned matrix S_clean (924 x 3401).'''

In [ ]:
#Cell 10 — Question 3a, part (ii): the other error (high-norm outliers)

# Besides the empty vials, the row-norm distribution shows a SECOND group far
# above the bulk, separated by a clear gap (~5x wider than any gap inside the
# bulk). These measurements form a contiguous block (rows 924-959) sitting
# directly in front of the empty vials (rows 960-999): the whole tail of the
# matrix is anomalous, so we flag them as the "other error".

sn   = np.sort(norms)
gaps = np.diff(sn)
mid  = len(sn) // 2
hi   = mid + int(np.argmax(gaps[mid:]))        # widest gap ABOVE the median
high_thr = 0.5 * (sn[hi] + sn[hi + 1])

high = norms > high_thr                         # mask: high-norm outliers
print(f"high-norm threshold : {high_thr:.3f}")
print(f"outliers found      : {int(high.sum())}  "
      f"(rows {np.where(high)[0].min()}-{np.where(high)[0].max()})")
print(f"norm range          : {norms[high].min():.2f}-{norms[high].max():.2f}"
      f"  (bulk median {np.median(norms):.2f})")

In [ ]:
#Cell 11 — Produce the fully cleaned matrix (both errors removed)

# Keep a vial only if it is neither an empty blank nor a high-norm outlier.
keep = real & ~high
S_clean = S[keep]                               # overwrite: this is now final

print("removed - empty vials    :", int(empty.sum()))   # 40
print("removed - high-norm group:", int(high.sum()))    # 36
print("cleaned design matrix    :", S_clean.shape)       # (924, 3401)

In [ ]:
#Cell 12 — Evidence: are the 36 high-norm vials their own cluster?

# Concrete test of the part-(ii) decision. Cluster the empties-removed data in
# PCA space WITH vs WITHOUT the 36 high-norm vials. If they are a single
# distinct group, removing them should drop the cluster count by exactly one
# and leave every other cluster untouched.
import os

def pca_scores(X, k=4):
    """Top-k principal-component scores of X via its Gram matrix (X-mean)(X-mean)^T -- cheaper than the covariance when features outnumber samples. Returns an (n_samples, k) array; column j is the score on principal component j."""
    # project samples onto their top-k principal components (via the Gram matrix)
    Xc = X - X.mean(0)
    w, v = np.linalg.eigh(Xc @ Xc.T)
    idx = np.argsort(w)[::-1][:k]
    return v[:, idx] * np.sqrt(np.maximum(w[idx], 0.0))

def count_clusters(P, r):
    """Leader clustering (numpy-only, no sklearn): scan the points P and start a new cluster centre whenever a point lies farther than radius r from every existing centre. Returns the number of clusters found."""
    # leader clustering (numpy-only): start a new cluster when a point is
    # farther than r from every existing cluster centre
    centres = [P[0]]
    for p in P[1:]:
        if np.sqrt(((np.array(centres) - p) ** 2).sum(1)).min() > r:
            centres.append(p)
    return len(centres)

for label, mask in [("WITH    high-norm (960 vials)", real),
                    ("WITHOUT high-norm (924 vials)", real & ~high)]:
    P = pca_scores(S[mask], k=4)
    D = np.sqrt(((P[:, None, :] - P[None, :, :]) ** 2).sum(-1))
    np.fill_diagonal(D, np.inf)
    r = 8 * np.median(D.min(1))                  # radius ~ 8x typical neighbour spacing
    print(f"{label}: {count_clusters(P, r)} clusters")

# How isolated is the group, in the top-4 PC space?
P = pca_scores(S[real], k=4)
is_high = high[real]                             # which of the 960 kept vials are high-norm
ch     = P[is_high].mean(0)
radius = np.sqrt(((P[is_high]  - ch) ** 2).sum(1)).mean()
gap    = np.sqrt(((P[~is_high] - ch) ** 2).sum(1)).min()
print(f"\nhigh-norm cluster radius = {radius:.2f}, gap to nearest other vial = {gap:.2f} "
      f"({gap / radius:.0f}x its own radius)")
print("=> removing the 36 deletes exactly one well-separated cluster; the rest are untouched.")

# Visual: top-2 PCs with the 36 highlighted
plt.figure(figsize=(6, 5))
plt.scatter(P[~is_high, 0], P[~is_high, 1], s=10, alpha=0.4, label="other vials")
plt.scatter(P[is_high, 0],  P[is_high, 1],  s=30, color="red", label="high-norm group (36)")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend()
plt.title("The 36 high-norm vials form one isolated cluster")
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3a_part_ii_clusters.png", dpi=120)
plt.show()

In [ ]:
#Module purpose. Question 3b: per-feature variance and the covariance matrix.

'''Variance, mean-centering and the covariance matrix.
  Inputs. the cleaned design matrix S_clean (924 x 3401) from 3a.
  Method. Mean-center each feature (subtract its column mean) to get X_centered = X~. The
    per-feature variance is the diagonal of the covariance matrix C = (1/N~) X~^T X~, so we read
    it off directly as X_centered.var(axis=0) without ever forming the full 3401 x 3401 matrix.
    Convention: np.var divides by N~ (population variance), matching the definition of C.
  Sanity check. trace(C) = sum of variances = total variance (~7.90) = sum of eigenvalues (3c).
  Outputs. X_centered, N_tilde, the `variances` vector, j_star (=910); figure q3b_variance.png.'''

In [ ]:
#Cell 13 — Question 3b: mean-centre and per-feature variance

# Mean-centre each feature (column) so it has zero average across the N~ vials.
N_tilde = S_clean.shape[0]
X_centered = S_clean - S_clean.mean(axis=0)          # X~, shape (924, 3401)

# Covariance matrix C = (1/N~) X~^T X~  (M x M).  We only need its DIAGONAL here:
# C_jj = variance of feature j.  np.var divides by N~, matching the definition.
variances = X_centered.var(axis=0)                   # = diag(C)

# Sanity check: trace(C) = total variance (invariant under rotation/compression).
print("total variance  tr(C) =", round(variances.sum(), 4))

# 3b(i): the single feature with the largest variance
j_star = int(np.argmax(variances))
print(f"3b(i)  largest-variance feature  j* = {j_star}  "
      f"(variance = {variances[j_star]:.5f})")

In [ ]:
#Cell 14 — Question 3b(ii): plot the variance as a function of feature index

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(variances, lw=0.6, color="steelblue")
ax.axvline(j_star, color="red", ls="--", lw=1, label=f"j* = {j_star}  (largest-variance feature)")
ax.set_xlabel("feature index j  (frequency channel)")
ax.set_ylabel(r"variance  $C_{jj}$")
ax.set_title("Per-feature variance across the 3401 frequencies")
ax.legend()
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3b_variance.png", dpi=120)
plt.show()

In [ ]:
#Module purpose. Question 3c: eigendecomposition of C (PCA) and the eigenspectrum.

'''Principal Component Analysis via the covariance matrix.
  Inputs. X_centered, N_tilde (from 3b).
  Method. Form C = (1/N~) X~^T X~ and diagonalise it with numpy.linalg.eigh -- the symmetric
    solver: real eigenvalues, orthonormal eigenvectors, faster and stabler than eig. Sort the
    eigenvalues descending and clip rounding-level negatives to 0 (C is positive semi-definite).
  Analyses. (i) ratio lambda1/lambda2; (ii) the spectrum on linear and log scales; (iii) the
    elbow k* = the largest multiplicative drop between consecutive eigenvalues (k*=8); cumulative
    variance (95% needs 667 components); and a saved ratios table (PNG + CSV).
  Outputs. C, eigvals, eigvecs (feature-space PCs, reused in 3d/3e), k_star; figures
    q3c_spectrum.png, q3c_cvarr.png, q3c_ratios.png and the table q3c_ratios.csv.'''

In [ ]:
#Cell 16 — Question 3c: eigendecomposition of the covariance matrix (PCA)

# Covariance matrix  C = (1/N~) X~^T X~   (M x M = 3401 x 3401).
C = (X_centered.T @ X_centered) / N_tilde

# eigh is for SYMMETRIC matrices: it guarantees real eigenvalues and an
# orthonormal eigenvector basis, and is faster/more stable than eig.
# It returns them in ASCENDING order, so we reverse to largest-first.
w, V = np.linalg.eigh(C)
order   = np.argsort(w)[::-1]
eigvals = np.clip(w[order], 0, None)    # clip tiny negatives caused by round-off
eigvecs = V[:, order]                   # columns = principal directions nu_k (feature space)

# Sanity check: sum of eigenvalues == trace(C) == total variance.
print("sum(eigenvalues) =", round(float(eigvals.sum()), 4),
      " vs  tr(C) =", round(float(C.trace()), 4))

# 3c(i): ratio of the two largest eigenvalues
print(f"3c(i)  lambda1/lambda2 = {eigvals[0] / eigvals[1]:.3f}")

In [ ]:
#Cell 17 — Question 3c(ii)+(iii): eigenvalue spectrum (linear + log) and the elbow

# 3c(iii): locate the elbow = the largest multiplicative drop between
# consecutive eigenvalues in the leading part of the spectrum.
ratios = eigvals[:30] / eigvals[1:31]
k_star = int(np.argmax(ratios)) + 1             # number of "signal" eigenvalues
print(f"3c(iii)  elbow at k* = {k_star}  "
      f"(drop lambda{k_star}/lambda{k_star+1} = {ratios[k_star-1]:.2f})")

top = 50                                         # show the leading spectrum
k = np.arange(1, top + 1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for a, scale in zip(ax, ["linear", "log"]):
    a.plot(k, eigvals[:top], "o-", ms=4, color="steelblue")
    a.axvline(k_star + 0.5, color="red", ls="--", label=f"elbow k* = {k_star}")
    a.set_yscale(scale)
    a.set_xlabel("component index k")
    a.set_ylabel(r"eigenvalue $\lambda_k$")
    a.set_title(f"Eigenvalue spectrum ({scale} scale)")
    a.legend()
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3c_spectrum.png", dpi=120)
plt.show()